# Improve PDF chunking: 

Let's look at how currently chunking is done. 

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd


def _project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "wellground"
        ).is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find wellground repo root from {start}")


# Settings and chunk files are resolved from the repo root (idempotent if re-run).
PROJECT_ROOT = _project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

from wellground.config import get_settings

settings = get_settings()
chunks_path = (PROJECT_ROOT / settings.wellground_bm25_path / "chunks.jsonl").resolve()

# Chunks are stored as JSON Lines (one record per line).
df = pd.read_json(chunks_path, lines=True)
print(f"Loaded {len(df)} chunks from {chunks_path}")
df.head(10)

if "section" in df.columns:
    print(df["section"].value_counts(dropna=False).head(20))


Loaded 594 chunks from /Users/mehuljain/Documents/Projects/well/data/processed/bm25/chunks.jsonl
section
Body                                                 533
University of Utah                                    15
Header                                                 4
Table of Abbreviations                                 4
Tool Mnemonics List                                    4
Data/Sensor Mnemonics List                             4
Interpretation Mnemonics List                          4
References                                             4
End of Report                                          4
LEGAL DISCLAIMER                                       3
RELATING TO THE DRILLING OR PRODUCTION OPERATION.      3
Objectives                                             2
Well / Job Information                                 2
General Logging Procedure                              2
Interpretation Remarks                                 2
PSP Interpretation Results              

In [2]:
df

,chunk_id,doc_id,title,page,text,well_ids,token_count,section
0,daily_reports/extracted/FORGE 16A(78)-32 Circu...,daily_reports/extracted/FORGE 16A(78)-32 Circu...,"FORGE 16A(78)-32 Circulation Test RPT No.1, 8-...",1,## Body\n| | | | | Daily Report Utah FORGE...,"[16A, 16B]",600,Body
1,daily_reports/extracted/FORGE 16A(78)-32 Circu...,daily_reports/extracted/FORGE 16A(78)-32 Circu...,"FORGE 16A(78)-32 Circulation Test RPT No.1, 8-...",1,## Body\n| | | | | Last BOP Test: | | | ...,"[16A, 16B]",600,Body
2,daily_reports/extracted/FORGE 16A(78)-32 Circu...,daily_reports/extracted/FORGE 16A(78)-32 Circu...,"FORGE 16A(78)-32 Circulation Test RPT No.1, 8-...",1,## Body\n| | | | | | | | | | | | |...,"[16A, 16B]",598,Body
3,daily_reports/extracted/FORGE 16A(78)-32 Circu...,daily_reports/extracted/FORGE 16A(78)-32 Circu...,"FORGE 16A(78)-32 Circulation Test RPT No.1, 8-...",1,## Body\n| RIGU | Move in Liberty pumping equi...,"[16A, 16B]",599,Body
4,daily_reports/extracted/FORGE 16A(78)-32 Circu...,daily_reports/extracted/FORGE 16A(78)-32 Circu...,"FORGE 16A(78)-32 Circulation Test RPT No.1, 8-...",1,## Body\nTreatments: 0 | | | | | | | | ...,"[16A, 16B]",429,Body
...,...,...,...,...,...,...,...,...
589,inj_prod/16B/University of Utah_Forge16B(78)-3...,inj_prod/16B/University of Utah_Forge16B(78)-3...,University of Utah_Forge16B(78)-32_SLB_28Aug20...,9,## Data/Sensor Mnemonics List\n\nCALI_FSI Flow...,[16B],371,Data/Sensor Mnemonics List
590,inj_prod/16B/University of Utah_Forge16B(78)-3...,inj_prod/16B/University of Utah_Forge16B(78)-3...,University of Utah_Forge16B(78)-32_SLB_28Aug20...,9,## Interpretation Mnemonics List\n\nQGD/QGZT G...,[16B],120,Interpretation Mnemonics List
591,inj_prod/16B/University of Utah_Forge16B(78)-3...,inj_prod/16B/University of Utah_Forge16B(78)-3...,University of Utah_Forge16B(78)-32_SLB_28Aug20...,10,## University of Utah\n\nForge 16B(78)-32,[16B],13,University of Utah
592,inj_prod/16B/University of Utah_Forge16B(78)-3...,inj_prod/16B/University of Utah_Forge16B(78)-3...,University of Utah_Forge16B(78)-32_SLB_28Aug20...,10,"## References\n\nFor more information, please ...",[16B],180,References


In [3]:
df['text'].iloc[10]

'## Body\n|  |  |  |  | Daily Report Utah FORGE Well ID:16A(78)-32-STIM1 Job ID: Circulation Test Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United States |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |\n| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |\n| Report No: 11 Report For 06:00 AM 16-Aug-24 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |\n| Operator: U of U / FORGE Project |  |  |  |  |  |  |  |  |  |  | Rig: |  |  |  | Spud Date: 12-Apr-22 |  |  |  | Daily Cost / Mud ($): --- |  |  |  |  |  |  |  |\n| Measured Depth (ft): 10987.0 |  |  |  |  |  |  |  |  |  |  | Last Casing: |  |  |  | Wellbore: Original Wellbore |  |  |  | AFE No. AFE ($) Actual ($) |  |  |  |  |  |  |  |\n| Vertical Depth (ft): 8559.0 |  |  |  |  |  |  |  |  |  |  | Next Casing: |  |  |  | RKB Elevation (ft)